# what distortion results from projecting episode into alternate spaces?

### imports

In [25]:
import numpy as np
import pandas as pd
import hypertools as hyp
import numpy as np
import os
import re
from num2words import num2words
from scipy.spatial.distance import cdist
from scipy.signal import resample
from scipy.stats import zscore
from scipy.spatial.distance import correlation
from scipy.interpolate import interp1d as interpolate
%matplotlib inline

### paths to data dirs

In [2]:
annot_dir = '../../data/annotations_dfs/'
model_dir = '../../data/models/'
transc_dir = '../../data/transcriptions/automatic/'
pickle_dir = '../../data/pickles/'

## load annotations dataframes

In [3]:
atlep1_df = pd.read_pickle(annot_dir+'atlep1.p')
atlep2_df = pd.read_pickle(annot_dir+'atlep2.p')
arrdev_df = pd.read_pickle(annot_dir+'arrdev.p')

## model parameters

In [4]:
n_topics = 100
episode_wsize = 50
recall_wsize = 200

# vectorizer parameters
vectorizer_params = {
    'model' : 'CountVectorizer', 
    'params' : {
        'stop_words' : 'english'
    }
}

# topic model parameters
semantic_params = {
    'model' : 'LatentDirichletAllocation', 
    'params' : {
        'n_components' : n_topics,
        'learning_method' : 'batch',
        'random_state' : 0,
    }
}

## functions for topic modeling and interpolation

In [5]:
def format_episode_text(textlist):
    """
    takes care of standardizing annotation text format for modeling
    """
    new_textlist = []
    
    for chunk in textlist:
        
        new_chunks = []
        for subchunk in chunk.split(','):
            # remove all characters except letters, spaces, apostrophes, dashes. Put all in uppercase
            upper_nopunc = re.sub("[^\w\s'-]+", '', subchunk.upper())

            # remove accented characters
            no_acc = upper_nopunc.replace('É', 'E')

            # convert digits to numbers
            if any(word.isdigit() for word in no_acc.split(' ')):
                for word in no_acc.split(' '):
                    if word.isdigit():
                        no_acc = no_acc.replace(word, num2words(int(word)).upper())
                    
            new_chunks.append(no_acc)
        
        new_textlist.append(','.join(new_chunks))
    
    return new_textlist

In [6]:
def get_episode_windows(episode_df, episode_wsize=episode_wsize):
    # throw all annotations into bag of words to train model
    episode_bag = format_episode_text(episode_df.loc[:,'Narrative details (external events)':'Setting'].apply(
        lambda x: ', '.join(x.fillna('')), axis=1).values.tolist())

    # create list for annotation sliding windows (of size w_size)
    episode_w = []
    for idx, sentence in enumerate(episode_bag):
        episode_w.append(','.join(episode_bag[idx:idx+episode_wsize]))

    return episode_w

In [7]:
def find_midpoint_time(df, endframe_time):
    """
    returns list of timepoints at middle of each annotation segment
    """
    midpoint_times = []
    for i, tpt in enumerate(df['Onset time']):
        if i != len(df['Onset time'])-1:
            midpoint_time = np.mean([tpt, df['Onset time'][i+1]])
        else:
            midpoint_time = np.mean([tpt, endframe_time])
        midpoint_times.append(midpoint_time)

    return midpoint_times

In [8]:
def interpolate_model(model, df, endframe_time, resolution=1):

    """
    uses linear interpolation to resample episode model timeseries to desired resolution. 
    'resolution' is in units of seconds (default is 1s).
    """
    
    # get middle timepoint for each annotation
    midpoint_times = find_midpoint_time(df, endframe_time)
    
    new_model = np.empty((int(round(endframe_time)),np.shape(model)[1]))
    
    # loop over topic dimensions
    for dim in range(np.shape(model)[1]):
        # values for given dimension at each timepoint
        single_dim = []
        for tpt in range(np.shape(model)[0]):
            single_dim.append(model[tpt][dim])
        
        # create interpolation function from dimension timeseries
        interp_func = interpolate(midpoint_times, single_dim, fill_value='extrapolate')
        
        # set of new timepoints
        new_tpts = np.arange(int(round(endframe_time)), step=resolution)
        
        # interpolate single topic dimension trajectory new timescale
        single_dim_res = interp_func(new_tpts)
        
        # fill in array for resampled model
        for ix, new_tpt in enumerate(single_dim_res):
            new_model[ix][dim] = new_tpt
    
    return new_model

In [9]:
def model_and_transform(documents, resample_shape=None, n_topics=n_topics, vec_params=vectorizer_params, 
                        sem_params=semantic_params, corpus=None):
    
    # if an episode dataframe is passed, create overlapping text windows from annotations
    if type(documents) is pd.core.frame.DataFrame:
        wsize = episode_wsize
        windows = get_episode_windows(documents, wsize)
    
        # if no corpus is passed, project episode into self-defined representational space
        if corpus == None:
            corpus = windows
        
    # if a transcript is passed, create overlapping windows from speech segments
    elif type(documents) is str:
        wsize = recall_wsize
        windows = get_recall_windows(documents, wsize)
            
    # use hypertools to create episode model
    model =  hyp.tools.format_data(windows, vectorizer=vec_params, semantic=sem_params, corpus=corpus)[0]
    
    # resample episode model using interpolation and corresponding endframe time
    if type(documents) is pd.core.frame.DataFrame:
        
        if np.shape(documents) == np.shape(atlep1_df):
            endframe_time = 1466.0
        elif np.shape(documents) == np.shape(atlep2_df):
            endframe_time = 1316.52
        elif np.shape(documents) == np.shape(arrdev_df):
            endframe_time = 1236.6
            
        return interpolate_model(model, documents, endframe_time)
    
    # resample recall model to shape of corresponding episode model
    elif type(documents) is str:
        return resample(model, resample_shape)

## create corpus for each topic space

In [135]:
atlep1_corpus = get_episode_windows(atlep1_df)

atlep2_corpus = get_episode_windows(atlep2_df)

arrdev_corpus = get_episode_windows(arrdev_df)

all_eps_corpus = atlep1_windows + atlep2_windows + arrdev_windows

wiki_docs = hyp.load('wiki').get_data()
wiki_corpus = []
for article in range(np.shape(wiki_docs)[1]):
    wiki_corpus.append(wiki_docs[0][article][0].decode('utf8').replace('\n',' '))

In [131]:
wiki_corpus_model = model_and_transform(atlep1_df, corpus=wiki_pages_corpus)